In [5]:
import time
import urllib.request
import numpy as np
import pandas as pd
import yfinance as yf

from valuation_utils import fetch_growth_features


def get_nifty500_tickers() -> list[str]:
    url = "https://archives.nseindia.com/content/indices/ind_nifty500list.csv"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    try:
        req = urllib.request.Request(url, headers=headers)
        with urllib.request.urlopen(req, timeout=10) as response:
            with open("nifty500_raw_list.csv", "wb") as f:
                f.write(response.read())
            
        df_tickers = pd.read_csv("nifty500_raw_list.csv")
        symbols = df_tickers['Symbol'].dropna().unique().tolist()
        tickers_yf = [f"{sym.strip()}.NS" for sym in symbols]
        print(f"Successfully fetched {len(tickers_yf)} Nifty 500 tickers from NSE.")
        return tickers_yf
    except Exception as e:
        print(f"Error fetching from NSE ({e}). Using fallback tickers...")
        fallback = ["RELIANCE", "TCS", "HDFCBANK", "INFY", "ICICIBANK", "HINDUNILVR", "ITC", "SBIN", "BHARTIARTL", "LTIM"]
        return [f"{sym}.NS" for sym in fallback]


def fetch_financial_data(ticker_symbol: str) -> dict | None:
    try:
        ticker = yf.Ticker(ticker_symbol)
        info = ticker.info
        if not info or ('regularMarketPrice' not in info and 'currentPrice' not in info):
            return None

        return {
            'Ticker': ticker_symbol,
            'Company_Name': info.get('shortName', info.get('longName', 'N/A')),
            'Sector': info.get('sector', 'Unknown'),
            'Industry': info.get('industry', 'Unknown'),
            'Market_Cap': info.get('marketCap', np.nan),
            'Current_Price': info.get('currentPrice', info.get('regularMarketPrice', np.nan)),
            'Trailing_PE': info.get('trailingPE', np.nan),
            'Forward_PE': info.get('forwardPE', np.nan),
            'Price_To_Book': info.get('priceToBook', np.nan),
            'Price_To_Sales': info.get('priceToSalesTrailing12Months', np.nan),
            'EV_To_EBITDA': info.get('enterpriseToEbitda', np.nan),
            'ROE': info.get('returnOnEquity', np.nan),
            'ROA': info.get('returnOnAssets', np.nan),
            'Operating_Margin': info.get('operatingMargins', np.nan),
            'Net_Profit_Margin': info.get('profitMargins', np.nan),
            'Revenue_Growth': info.get('revenueGrowth', np.nan),
            'Earnings_Growth': info.get('earningsGrowth', np.nan),
            'Debt_To_Equity': info.get('debtToEquity', np.nan),
            'Current_Ratio': info.get('currentRatio', np.nan),
            'Free_Cash_Flow': info.get('freeCashflow', np.nan),
            'Total_Revenue': info.get('totalRevenue', np.nan)
        }
    except Exception:
        return None


def build_and_clean_dataset():
    tickers = get_nifty500_tickers()
    dataset = []
    
    print(f"\nStarting financial data download for {len(tickers)} companies...")
    start_time = time.time()
    
    for idx, ticker in enumerate(tickers, start=1):
        print(f"[{idx}/{len(tickers)}] Downloading {ticker}...", end="\r")
        data = fetch_financial_data(ticker)
        if data:
            dataset.append(data)
        time.sleep(0.3)

    base_df = pd.DataFrame(dataset)
    base_df.dropna(subset=['Market_Cap', 'Current_Price'], inplace=True)

    print("\nFetching growth and risk metrics...")
    growth_rows = []
    for idx, t in enumerate(base_df['Ticker'], 1):
        print(f"[{idx}/{len(base_df)}] Growth metrics {t}", end="\r")
        growth_rows.append(fetch_growth_features(t))
        time.sleep(0.3)

    growth_df = pd.DataFrame(growth_rows)
    merged_df = base_df.merge(growth_df, on='Ticker', how='left')

    cleaned_df = merged_df.drop_duplicates('Ticker')
    n_initial = len(cleaned_df)
    cleaned_df = cleaned_df[(cleaned_df['Market_Cap'] > 0) & (cleaned_df['Current_Price'] > 0)]
    cleaned_df = cleaned_df[cleaned_df['Trailing_PE'].notna() & (cleaned_df['Trailing_PE'] > 0)]
    
    cleaned_df.to_csv("nifty500_clean_v2.csv", index=False)
    elapsed = round((time.time() - start_time) / 60, 2)
    print(f"\nDataset successfully saved to 'nifty500_clean_v2.csv'!")
    print(f"Valid records retained: {len(cleaned_df)} / {n_initial} (Dropped {n_initial - len(cleaned_df)} without valid P/E).")
    print(f"Time elapsed: {elapsed} minutes.")

build_and_clean_dataset()

Successfully fetched 501 Nifty 500 tickers from NSE.

Starting financial data download for 501 companies...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DUMMYHEG.NS"}}}


C:\Users\dpate\AppData\Local\Temp\ipykernel_7312\1733717898.py:81: Pandas4Warning: The inplace keyword in DataFrame.dropna is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  base_df.dropna(subset=['Market_Cap', 'Current_Price'], inplace=True)



Fetching growth and risk metrics...
[497/497] Growth metrics ECLERX.NS.NSS
Dataset successfully saved to 'nifty500_clean_v2.csv'!
Valid records retained: 474 / 497 (Dropped 23 without valid P/E).
Time elapsed: 13.45 minutes.
